[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/43_dapo_loss_solution.ipynb)

# 参考解答：解耦裁剪与动态采样策略优化损失（DAPO）

参考解答。

In [ ]:
# 在 Colab 中安装 torch-judge（JupyterLab/Docker 中会跳过）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn.functional as F
from torch import Tensor

In [ ]:
# ✅ 参考解答

def dapo_loss(new_logps: Tensor,
              old_logps: Tensor,
              rewards: Tensor,
              group_ids: Tensor,
              mask: Tensor | None = None,
              clip_low: float = 0.2,
              clip_high: float = 0.28,
              eps: float = 1e-6) -> Tensor:
    """解耦裁剪与动态采样策略优化（DAPO）损失。

    new_logps: (B, T) 当前策略的词元级对数概率
    old_logps: (B, T) 旧策略的词元级对数概率
    rewards: (B,) 每条采样回复的标量奖励
    group_ids: (B,) 整数组编号，相同编号表示同一个提示词／组
    mask: 可选，形状为 (B, T) 的词元掩码，1 表示计入，0 表示忽略
    clip_low: 下裁剪偏移量（下界为 1 - clip_low）
    clip_high: 上裁剪偏移量（上界为 1 + clip_high）
    eps: 优势标准化中的数值稳定项
    返回：标量损失（Tensor）
    """
    r_detached = rewards.detach().to(dtype=new_logps.dtype, device=new_logps.device)
    advantages = torch.zeros_like(r_detached)
    valid_seq = torch.zeros_like(r_detached)

    for gid in group_ids.unique():
        g_mask = group_ids == gid
        r_g = r_detached[g_mask]
        std_g = r_g.std(unbiased=False)
        if std_g > 1e-12:
            mean_g = r_g.mean()
            advantages[g_mask] = (r_g - mean_g) / (std_g + eps)
            valid_seq[g_mask] = 1.0

    if mask is None:
        token_mask = torch.ones_like(new_logps)
    else:
        token_mask = mask.to(dtype=new_logps.dtype, device=new_logps.device)

    active_mask = token_mask * valid_seq.unsqueeze(-1)
    total_active_tokens = active_mask.sum()
    if total_active_tokens <= 0:
        return new_logps.sum() * 0.0

    # 仅在有效词元位置计算重要性采样比率，避免填充位极端数值溢出
    log_diff = torch.where(
        active_mask > 0,
        new_logps - old_logps.detach(),
        torch.zeros_like(new_logps),
    )
    ratios = torch.exp(log_diff)
    adv_2d = advantages.detach().unsqueeze(-1)

    unclipped = ratios * adv_2d
    clipped = torch.clamp(ratios, 1.0 - clip_low, 1.0 + clip_high) * adv_2d
    surrogate = torch.min(unclipped, clipped)

    return -(surrogate * active_mask).sum() / total_active_tokens

In [ ]:
# 示例
ratios = torch.tensor([[1.25, 1.40], [0.75, 0.85]])
old_logps = torch.zeros(2, 2)
new_logps = ratios.log().requires_grad_(True)
rewards = torch.tensor([1.0, -1.0])
group_ids = torch.tensor([0, 0])
print('Loss:', dapo_loss(new_logps, old_logps, rewards, group_ids, eps=0.0).item())

In [ ]:
from torch_judge import check
check('dapo_loss')